# Polymarket analysis
Credentials come from `notebooks/.env`. Run the setup cell once, then use:

- `ask("SQL")` -> catalog tables `markets`, `events`, `resolutions` (Postgres)
- `book("SQL")` -> parquet files: `{BASE}/snap-<hour>.parquet`, `{BASE}/events-<hour>.parquet`, `{BASE}/wallet_trades/trades-<day>.parquet`

Both return a pandas DataFrame. `.head(10)` shows the first ten rows. Access closes Oct 29, 11:59pm PDT; save anything you need with `df.to_parquet(...)`.

In [1]:
import os, json, time
import pandas as pd, duckdb
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)

load_dotenv("notebooks/.env")
engine = create_engine(os.environ["JUNE_DB_URL"], connect_args={"sslmode": "require"})
BASE = f"https://{quote(os.environ['BOOK_USER'], safe='')}:{quote(os.environ['BOOK_PASSWORD'], safe='')}@d1yyqbluk0oz0t.cloudfront.net"
duckdb.sql("INSTALL httpfs; LOAD httpfs;")

def ask(sql, **params):
    """Query the catalog (Postgres). Returns a DataFrame."""
    with engine.connect() as con:
        return pd.read_sql(text(sql), con, params=params or None)

def book(sql):
    """Query the parquet files (DuckDB). Returns a DataFrame."""
    return duckdb.sql(sql).df()

print("ready")

ready


## Catalog: first ten markets

In [2]:
markets = ask("SELECT slug, title, best_bid, best_ask, volume_24hr, end_date FROM markets ORDER BY volume_24hr DESC NULLS LAST LIMIT 10")
markets.head(10)

,slug,title,best_bid,best_ask,volume_24hr,end_date
0,fifwc-esp-arg-2026-07-19-team-to-advance,Spain vs. Argentina: Team to Advance,0.9975,1.000,3.987195e+07,2026-07-19 19:00:00+00:00
1,fifwc-bel-egy-2026-06-15-bel,Will Belgium win on 2026-06-15?,NaN,0.001,2.649406e+07,2026-06-15 19:00:00+00:00
2,will-egypt-win-the-2026-fifa-world-cup,Will Egypt win the 2026 FIFA World Cup?,NaN,0.001,2.536157e+07,2026-07-20 00:00:00+00:00
3,microstrategy-sells-any-bitcoin-by-may-31-2026,"MicroStrategy sells any Bitcoin by May 31, 2026?",NaN,0.001,2.492871e+07,2026-07-01 04:00:00+00:00
4,us-x-iran-permanent-peace-deal-by-june-15-2026-734-856-129,"US x Iran permanent peace deal by June 15, 2026?",0.8370,0.847,2.292461e+07,2026-06-15 00:00:00+00:00
5,will-cape-verde-win-the-2026-fifa-world-cup,Will Cape Verde win the 2026 FIFA World Cup?,NaN,0.001,2.229070e+07,2026-07-20 00:00:00+00:00
6,will-spain-win-the-2026-fifa-world-cup-963,Will Spain win the 2026 FIFA World Cup?,0.9990,1.000,2.157227e+07,2026-07-20 00:00:00+00:00
7,atp-cobolli-zverev-2026-06-07,Roland Garros ATP: Flavio Cobolli vs Alexander Zverev,NaN,0.001,1.798970e+07,2026-06-14 13:00:00+00:00
8,fifwc-uzb-col-2026-06-17-col,Will Colombia win on 2026-06-17?,0.9830,0.986,1.730192e+07,2026-06-18 02:00:00+00:00
9,will-france-win-the-2026-fifa-world-cup-924,Will France win the 2026 FIFA World Cup?,NaN,0.001,1.607742e+07,2026-07-20 00:00:00+00:00


## Catalog: what columns exist

In [3]:
ask("SELECT column_name, data_type FROM information_schema.columns WHERE table_name = 'markets' ORDER BY ordinal_position")

,column_name,data_type
0,slug,text
1,title,text
2,condition_id,text
3,synced_at,bigint
4,start_date,timestamp with time zone
5,end_date,timestamp with time zone
6,active,boolean
7,closed,boolean
8,archived,boolean
9,created_at,timestamp with time zone


## Order book: first ten level changes in one hour (UTC)

In [4]:
HOUR = "2026-09-29T17"
events = book(f"SELECT * FROM '{BASE}/events-{HOUR}.parquet' LIMIT 10")
events.head(10)

,ts_ms,asset_id,market,side,price,size
0,1790701200057,71677612305244936641085500097737038199081845621961106192...,0xaece8d6062d3e3eb9845b32441a4ab06eb2ef332c0228bc6a36add...,BUY,0.969,25.00
1,1790701200057,73197441127256680134600821323583356037261213281680365433...,0xaece8d6062d3e3eb9845b32441a4ab06eb2ef332c0228bc6a36add...,SELL,0.031,25.00
2,1790701200059,71677612305244936641085500097737038199081845621961106192...,0xaece8d6062d3e3eb9845b32441a4ab06eb2ef332c0228bc6a36add...,BUY,0.969,25.00
3,1790701200059,73197441127256680134600821323583356037261213281680365433...,0xaece8d6062d3e3eb9845b32441a4ab06eb2ef332c0228bc6a36add...,SELL,0.031,25.00
4,1790701200091,11328770156420933991369334740568574998628599914635237526...,0x307a1ed89d60b61002dd5bbf00e1408c5ed2ab3fcdb056191ca7ef...,BUY,0.620,70644.64
5,1790701200091,10711843368565070233204099145407890218847374659922373290...,0x307a1ed89d60b61002dd5bbf00e1408c5ed2ab3fcdb056191ca7ef...,SELL,0.380,70644.64
6,1790701200096,11328770156420933991369334740568574998628599914635237526...,0x307a1ed89d60b61002dd5bbf00e1408c5ed2ab3fcdb056191ca7ef...,BUY,0.620,70644.64
7,1790701200096,10711843368565070233204099145407890218847374659922373290...,0x307a1ed89d60b61002dd5bbf00e1408c5ed2ab3fcdb056191ca7ef...,SELL,0.380,70644.64
8,1790701200136,15077447585111445582636234598999564675487634195225264623...,0xdbe93b5a701f36076a560fa4b9ba59e365a6e8e2ea6a8376464001...,BUY,0.478,420.00
9,1790701200136,52487256270918227930942830546248720603566203568229607048...,0xdbe93b5a701f36076a560fa4b9ba59e365a6e8e2ea6a8376464001...,SELL,0.522,420.00


## Wallet trades: first ten fills on one day

In [5]:
DAY = "2026-10-05"
trades = book(f"SELECT * FROM '{BASE}/wallet_trades/trades-{DAY}.parquet' LIMIT 10")
trades.head(10)

,wallet,market,time,side,outcome,price,size,dollars
0,0xbbecb03a1e605571630e08545809358781ae323d,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-05 01:21:38-07:00,BUY,Yes,0.302231,244.71,73.9589
1,0xbbecb03a1e605571630e08545809358781ae323d,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-05 01:08:07-07:00,BUY,Yes,0.290000,105.61,30.6269
2,0x1b652cb2f344c4f4d24e6cd3b25a1618367ba402,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-04 21:33:10-07:00,BUY,Yes,0.230000,10.83,2.4909
3,0x6f28458f2ce48fadfcbdd4f707081a2cbdf0ad56,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-04 20:41:35-07:00,SELL,Yes,0.210000,10.83,2.2743
4,0x6f28458f2ce48fadfcbdd4f707081a2cbdf0ad56,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-04 20:31:34-07:00,SELL,Yes,0.230000,8.93,2.0539
5,0x5ba7e5926389b126cc63e8d9b3401e3ff7e20c73,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-04 20:24:37-07:00,BUY,No,0.730000,160.00,116.8000
6,0xd3b1221dad8f3f2a3145c34d3f7e901fd6165e13,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-04 20:04:35-07:00,SELL,Yes,0.270000,10.00,2.7000
7,0x6f28458f2ce48fadfcbdd4f707081a2cbdf0ad56,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-04 17:40:47-07:00,SELL,Yes,0.310000,60.24,18.6744
8,0x3fad9c448902a4baaf289902e0aa5d762a17b82d,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-04 17:40:44-07:00,BUY,No,0.689596,229.00,157.9175
9,0x890baf2748f09e19917d1f133ea740dc73339b45,0-ships-transit-hormuz-on-any-date-by-october-31,2026-10-04 17:20:46-07:00,SELL,Yes,0.320000,27.00,8.6400


## Save a slice to disk (optional)
Uncomment to keep a local copy you can analyze after Oct 29.

In [6]:
# os.makedirs("data", exist_ok=True)
# ask("SELECT * FROM markets").to_parquet("data/markets.parquet", index=False)
# duckdb.sql(f"COPY (SELECT * FROM '{BASE}/wallet_trades/trades-{DAY}.parquet') TO 'data/trades-{DAY}.parquet' (FORMAT PARQUET)")
# pd.read_parquet("data/markets.parquet").head(10)